In [0]:
## using widgets

dbutils.widgets.text("batch_id", "1", "Batch ID (1, 2, or 3)")


In [0]:
spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("create schema if not exists bronze")
spark.sql("use schema staging")

In [0]:
## initilixe variables
batch_id = dbutils.widgets.get("batch_id")

team_name = "team_lemma"
bronze_db = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"
Batch_Folder = f"Batch{batch_id}"


In [0]:
## Extract the carried run_id from the bronze_table

try:
    run_info_now = spark.sql(f"""
                               select _run_id , _batch FROM {bronze_db}.finwire 
                               where _batch = '{batch_id}'
                               LIMIT 1 
                            """).first()
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id


print(f"bronze  : {bronze_db}")
print(f"staging : {staging_db}")
run_id = carried_run_id
print(f"run_id  : {run_id}")

In [0]:

if batch_id != "1":
    print(f"Finwire batch 1 only not requirw for {batch_id}")
    dbutils.notebook.exit("Finwire batch 1 only ")

In [0]:
from pyspark.sql.functions import col, lit,current_timestamp , trim , substring , to_date , to_timestamp , try_to_timestamp

In [0]:
## read from bronze finwire

df_bronze = spark.table(f"{bronze_db}.finwire")

# excludes the blank/header lines

df_bronze = df_bronze\
              .filter(col("_batch") == "1")\
              .filter(col("raw_line").isNotNull())\
              .filter(col("raw_line") != "")\
              .filter(col("raw_line").cast("string").rlike("^.{18,}"))\
              .filter(col("raw_line").cast("string").contains("CMP") |
              (col("raw_line").cast("string").contains("SEC"))|
              (col("raw_line").cast("string").contains("FIN")))

bronze_count = df_bronze.count()
print(f"bronze count : {bronze_count}")




In [0]:
from pyspark.sql.functions import expr

def safe_date(column_name: str, fmt: str = "yyyyMMdd"):
    return expr(f"try_to_date(trim({column_name}), '{fmt}')")


def safe_eff_date():
    return expr("try_to_date(trim(substring(PTS, 1, 8)), 'yyyyMMdd')").alias("EffectiveDate")

In [0]:
## parse CMP record

df_cmp = df_bronze.filter(trim(substring(col("raw_line") , 16 , 3)) == "CMP")\
    .select(
           expr("try_to_timestamp(trim(substring(raw_line, 1, 15)), 'yyyyMMdd-HHmmss')").alias("PTS"),
            trim(substring(col("raw_line"), 19, 60)).alias("CompanyName"),
            trim(substring(col("raw_line"), 79, 10)).alias("CIK"),
            trim(substring(col("raw_line"), 89,  4)).alias("Status"),
            trim(substring(col("raw_line"), 93,  2)).alias("IndustryID"),
            trim(substring(col("raw_line"), 95,  4)).alias("SPRating"),
            trim(substring(col("raw_line"), 99,  8)).alias("FoundingDate_str"),
            trim(substring(col("raw_line"), 107, 40)).alias("AddrLine1"),
            trim(substring(col("raw_line"), 147, 40)).alias("AddrLine2"),
            trim(substring(col("raw_line"), 187, 12)).alias("PostalCode"),
            trim(substring(col("raw_line"), 199, 25)).alias("City"),
            trim(substring(col("raw_line"), 224, 20)).alias("StateProvince"),
            trim(substring(col("raw_line"), 244, 24)).alias("Country"),
            trim(substring(col("raw_line"), 268, 46)).alias("CEOName"),
            trim(substring(col("raw_line"), 314, 100)).alias("Description")
            
    )\
        .withColumn("FoundingDate" , safe_date("FoundingDate_str"))\
        .withColumn("RecType", lit("CMP"))\
        .withColumn("_batch" , lit(batch_id))\
        .withColumn("_run_id" , lit(run_id))\
        .withColumn("_load_ts" , current_timestamp())\
        .drop("FoundingDate_str")



print(f"CMP records parsed : {df_cmp.count()}")
## parse SEC record



                  


In [0]:
### parse SEC records

df_sec_raw = df_bronze.filter(trim(substring(col("raw_line") , 16 , 3)) == "SEC")\
    .select(
           expr("try_to_timestamp(trim(substring(raw_line, 1, 15)), 'yyyyMMdd-HHmmss')").alias("PTS"),
            trim(substring(col("raw_line") , 19 , 15)).alias("symbol"),
            trim(substring(col("raw_line"), 34,   6)).alias("IssueType"),
            trim(substring(col("raw_line"), 40,   4)).alias("Status"),
            trim(substring(col("raw_line"), 44,  70)).alias("SecurityName"),
            trim(substring(col("raw_line"), 114,  6)).alias("ExID"),
            trim(substring(col("raw_line"), 120, 13)).alias("SharesOutstanding"),
            trim(substring(col("raw_line"), 133,  8)).alias("FirstTradeDate_str"),
            trim(substring(col("raw_line"), 141,  8)).alias("FirstTradeExchange_str"),
            trim(substring(col("raw_line"), 149, 12)).alias("Dividend"),
            trim(substring(col("raw_line"), 161, 60)).alias("CoNameOrCIK"),
        )\
        .withColumn("FirstTradeDate", safe_date("FirstTradeDate_str"))\
        .withColumn("FirstTradeExchange", safe_date("FirstTradeExchange_str"))\
        .drop("FirstTradeDate_str", "FirstTradeExchange_str")\
        .withColumn("_batch",   lit(batch_id))\
        .withColumn("_run_id",  lit(run_id))\
        .withColumn("_load_ts", current_timestamp()
        )



# split into SEC_CIK 

df_sec_cik = df_sec_raw.filter(col("CoNameOrCIK").rlike("^[0-9]+$"))\
    .withColumn("RecType" , lit("SEC_CIK"))


df_sec_name = df_sec_raw.filter(~col("CoNameOrCIK").rlike("^[0-9]+$"))\
    .withColumn("RecType" ,lit("SEC_NAME"))


print(f"SEC records parsed : {df_sec_raw.count()}")
print(f"SEC_CIK records parsed : {df_sec_cik.count()}")
print(f"SEC_NAME records parsed : {df_sec_name.count()}")



    
    

In [0]:
### parse FIN records

df_fin_raw = df_bronze.filter(trim(substring(col("raw_line") , 16 , 3)) == "FIN")\
    .select(
            expr("try_to_timestamp(trim(substring(raw_line, 1, 15)), 'yyyyMMdd-HHmmss')").alias("PTS"),
            trim(substring(col("raw_line"), 19,   4)).alias("FI_Year"),
            trim(substring(col("raw_line"), 23,   1)).alias("FI_Quarter"),
            trim(substring(col("raw_line"), 24,   8)).alias("QtrStartDate_str"),
            trim(substring(col("raw_line"), 32,   8)).alias("PostingDate_str"),
            trim(substring(col("raw_line"), 40,  17)).alias("Revenue"),
            trim(substring(col("raw_line"), 57,  17)).alias("Earnings"),
            trim(substring(col("raw_line"), 74,  12)).alias("EPS"),
            trim(substring(col("raw_line"), 86,  12)).alias("DilutedEPS"),
            trim(substring(col("raw_line"), 98,  12)).alias("Margin"),
            trim(substring(col("raw_line"), 110, 17)).alias("Inventory"),
            trim(substring(col("raw_line"), 127, 17)).alias("Assets"),
            trim(substring(col("raw_line"), 144, 17)).alias("Liabilities"),
            trim(substring(col("raw_line"), 161, 13)).alias("SharesOutstanding"),
            trim(substring(col("raw_line"), 174, 13)).alias("DilutedSharesOut"),
            trim(substring(col("raw_line"), 187, 60)).alias("CoNameOrCIK")
           
        )\
        .withColumn("QtrStartDate",
            safe_date("QtrStartDate_str"))\
        .withColumn("PostingDate",
            safe_date("PostingDate_str"))\
        .drop("QtrStartDate_str", "PostingDate_str")\
        .withColumn("_batch",   lit(batch_id))\
        .withColumn("_run_id",  lit(run_id))\
        .withColumn("_load_ts", current_timestamp()
    )
    

    # Split into FIN_COMPANYID
df_fin_cik = (
        df_fin_raw
        .filter(col("CoNameOrCIK").rlike("^[0-9]+$"))
        .withColumn("RecType", lit("FIN_COMPANYID"))
    )

df_fin_name = (
        df_fin_raw
        .filter(~col("CoNameOrCIK").rlike("^[0-9]+$"))
        .withColumn("RecType", lit("FIN_NAME"))
    )


print(f"FIN records : {df_fin_raw.count()}")
print(f"FIN_COMPANYID records : {df_fin_cik.count()}")
print(f"FIN_NAME      records : {df_fin_name.count()}")

In [0]:
from pyspark.sql import Row

In [0]:
### UNION all 5 RECTYPES and write in staging

recon_results = []

df_staging = df_cmp\
              .unionByName(df_sec_cik , allowMissingColumns=True)\
              .unionByName(df_sec_name , allowMissingColumns=True)\
              .unionByName(df_fin_cik , allowMissingColumns=True)\
              .unionByName(df_fin_name , allowMissingColumns = True)


source_count = df_staging.count()

print(f"Source count : {source_count}")
# DBTITLE 1,Write to staging




In [0]:
## write in delta table

df_staging.write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("RecType")\
    .saveAsTable(f"{staging_db}.finwire_parsed")

In [0]:
staging_count = spark.table(f"{staging_db}.finwire_parsed").count()

status = "Match" if  source_count == staging_count else "Mismatch"

print(f"Source rows  : {source_count}")
print(f"Staging rows : {staging_count}")
print(f"Status       : {status}")


In [0]:
print(f"\nBreakdown by RecType:")
spark.table(f"{staging_db}.finwire_parsed").groupBy("RecType").count().orderBy("RecType").show()

recon_results.append(Row(
        source_table = "finwire_parsed",
        batch_id     = f"Batch{batch_id}",
        source_count = source_count,
        target_count = staging_count,
        status       = status
))

In [0]:
%run ../../02_common_utils/operations

In [0]:
## log and display

recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if row.status in ("MATCH", "MISMATCH"):
        log_pipeline_recon(
            spark        = spark,
            run_id       = run_id,
            batch_id     = row.batch_id,
            domain       = "MARKET",
            table_name   = row.source_table,
            source_layer = "bronze",
            target_layer = "staging",
            source_count = row.source_count,
            target_count = row.target_count
        )

        log_audit_event(
            spark         = spark,
            run_id        = run_id,
            batch         = row.batch_id,
            layer         = "staging",
            table_name    = row.source_table,
            operation     = "OVERWRITE",
            rows_affected = row.target_count
        )

display(recon_df)
